# Cross-Dataset CNN Evaluation

This notebook evaluates the selected 1D-CNN under the strict cross-dataset protocol. Each direction trains on one main dataset, uses only that dataset's fold-0 validation participants for early stopping, and evaluates on the other dataset.

The target dataset is never used for model selection. Predictions are aggregated by participant before metrics are calculated.

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
MAG_ARRAY_PATH = PROCESSED / 'validated_acceleration_magnitude_windows_float32.npy'
METADATA_PATH = PROCESSED / 'validated_window_metadata.csv'
SPLITS_PATH = PROJECT_ROOT / 'data' / 'interim' / 'participant_splits.csv'
EPOCHS = 6
BATCH_SIZE = 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
magnitude_windows = np.load(MAG_ARRAY_PATH, mmap_mode='r')
metadata = pd.read_csv(METADATA_PATH)
splits = pd.read_csv(SPLITS_PATH)
metadata['label_binary'] = metadata['label'].map({'healthy': 0, 'stroke': 1}).astype(int)
print('Device:', DEVICE)
print('Magnitude windows:', magnitude_windows.shape)


Device: cpu
Magnitude windows: (18511, 500, 3)


In [2]:
def fold_roles(fold=0):
    role_map = splits[splits['fold'].eq(fold)].set_index('participant_key')['role']
    return metadata['participant_key'].map(role_map)


def normalization(train_indices):
    total = np.zeros(3, dtype=np.float64)
    total_sq = np.zeros(3, dtype=np.float64)
    count = 0
    for start in range(0, len(train_indices), 512):
        batch = np.asarray(magnitude_windows[train_indices[start:start + 512]], dtype=np.float32)
        total += batch.sum(axis=(0, 1))
        total_sq += np.square(batch).sum(axis=(0, 1))
        count += batch.shape[0] * batch.shape[1]
    mean = total / count
    std = np.sqrt(np.maximum(total_sq / count - mean ** 2, 1e-8))
    return mean.astype(np.float32), std.astype(np.float32)


class MagnitudeDataset(Dataset):
    def __init__(self, indices, mean, std, weights=None):
        self.indices = np.asarray(indices, dtype=np.int64)
        self.mean = mean.reshape(1, 3)
        self.std = std.reshape(1, 3)
        self.weights = np.ones(len(self.indices), dtype=np.float32) if weights is None else np.asarray(weights, dtype=np.float32)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        source_index = self.indices[item]
        signal = np.asarray(magnitude_windows[source_index], dtype=np.float32)
        signal = ((signal - self.mean) / self.std).T.copy()
        label = np.float32(metadata.iloc[source_index]['label_binary'])
        return torch.from_numpy(signal), torch.tensor(label), torch.tensor(self.weights[item]), torch.tensor(source_index)


class GaitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(3, 32, kernel_size=9, padding=4),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(128, 1))

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)


def predict_participants(model, loader):
    model.eval()
    rows = []
    with torch.no_grad():
        for signals, labels, _, indices in loader:
            probabilities = torch.sigmoid(model(signals.to(DEVICE))).cpu().numpy()
            for index, label, probability in zip(indices.numpy(), labels.numpy(), probabilities):
                rows.append({
                    'window_index': int(index),
                    'participant_key': metadata.iloc[int(index)]['participant_key'],
                    'dataset_id': metadata.iloc[int(index)]['dataset_id'],
                    'label_binary': int(label),
                    'probability': float(probability),
                })
    frame = pd.DataFrame(rows)
    return frame.groupby(['participant_key', 'dataset_id', 'label_binary'], as_index=False)['probability'].mean()


def metric_row(frame):
    y_true = frame['label_binary'].to_numpy()
    y_prob = frame['probability'].to_numpy()
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        'participants': len(frame),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'f1': f1_score(y_true, y_pred),
    }


In [3]:
roles = fold_roles(0)
results = []
prediction_frames = []
for train_dataset, test_dataset in [('voisard_2025', 'felius_2024'), ('felius_2024', 'voisard_2025')]:
    train_indices = np.flatnonzero((metadata['dataset_id'].eq(train_dataset) & roles.eq('training')).to_numpy())
    validation_indices = np.flatnonzero((metadata['dataset_id'].eq(train_dataset) & roles.eq('validation')).to_numpy())
    test_indices = np.flatnonzero(metadata['dataset_id'].eq(test_dataset).to_numpy())
    mean, std = normalization(train_indices)
    counts = metadata.iloc[train_indices].groupby('participant_key').size()
    participant_weights = metadata.iloc[train_indices]['participant_key'].map(1.0 / counts).to_numpy()
    class_counts = metadata.iloc[train_indices].groupby('label_binary').size()
    class_weights = metadata.iloc[train_indices]['label_binary'].map(len(train_indices) / (2.0 * class_counts)).to_numpy()
    weights = participant_weights * class_weights
    weights = weights / weights.mean()
    train_loader = DataLoader(MagnitudeDataset(train_indices, mean, std, weights), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    validation_loader = DataLoader(MagnitudeDataset(validation_indices, mean, std), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(MagnitudeDataset(test_indices, mean, std), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model = GaitCNN().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best_auc = -np.inf
    best_state = None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for signals, labels, batch_weights, _ in train_loader:
            optimizer.zero_grad()
            logits = model(signals.to(DEVICE))
            loss = (nn.functional.binary_cross_entropy_with_logits(logits, labels.to(DEVICE), reduction='none') * batch_weights.to(DEVICE)).mean()
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))
        validation_frame = predict_participants(model, validation_loader)
        validation_metrics = metric_row(validation_frame)
        print('train={} epoch={} loss={:.4f} internal_auc={:.3f}'.format(train_dataset, epoch, np.mean(losses), validation_metrics['roc_auc']))
        if validation_metrics['roc_auc'] > best_auc:
            best_auc = validation_metrics['roc_auc']
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    model.load_state_dict(best_state)
    test_frame = predict_participants(model, test_loader)
    test_metrics = metric_row(test_frame)
    results.append({
        'train_dataset': train_dataset,
        'test_dataset': test_dataset,
        'internal_validation_roc_auc': best_auc,
        **test_metrics,
    })
    test_frame['train_dataset'] = train_dataset
    test_frame['test_dataset'] = test_dataset
    prediction_frames.append(test_frame)
    torch.save({'model_state_dict': model.state_dict(), 'mean': mean, 'std': std, 'train_dataset': train_dataset, 'test_dataset': test_dataset}, PROCESSED / f'cnn_cross_dataset_{train_dataset}_to_{test_dataset}.pt')

results = pd.DataFrame(results)
predictions = pd.concat(prediction_frames, ignore_index=True)
print(results.round(3).to_string(index=False))
results.to_csv(PROCESSED / 'cnn_cross_dataset_results.csv', index=False)
predictions.to_csv(PROCESSED / 'cnn_cross_dataset_predictions.csv', index=False)


train=voisard_2025 epoch=1 loss=0.3458 internal_auc=1.000


train=voisard_2025 epoch=2 loss=0.2345 internal_auc=1.000


train=voisard_2025 epoch=3 loss=0.1874 internal_auc=1.000


train=voisard_2025 epoch=4 loss=0.1405 internal_auc=1.000


train=voisard_2025 epoch=5 loss=0.1075 internal_auc=1.000


...

train=felius_2024 epoch=3 loss=0.1804 internal_auc=0.889


train=felius_2024 epoch=4 loss=0.1820 internal_auc=0.912


train=felius_2024 epoch=5 loss=0.1385 internal_auc=0.923


train=felius_2024 epoch=6 loss=0.1174 internal_auc=0.881


train_dataset test_dataset  internal_validation_roc_auc  participants  balanced_accuracy  roc_auc    f1
 voisard_2025  felius_2024                        1.000           163              0.740    0.879 0.753
  felius_2024 voisard_2025                        0.923           121              0.528    0.509 0.516
